In [ ]:
# Analytic model, with parameters from yaml
# Keep all the model equations in this cell (called by shower_toy)

import numpy as np
import matplotlib.pyplot as plt
from model import alpha, gamma, epsilon, beta, eta, lmbda, C, phi_D, YEAR, load

print(f"alpha {alpha}  gamma {gamma}  epsilon {epsilon:g}  beta {beta}  eta {eta}  lambda {lmbda}  C {C}  "
      f"phi_D {phi_D}")

def dJdE(E):
    return alpha / epsilon * (E / epsilon)**(-gamma)

def J(E):
    """Intensity above E."""
    return alpha / (gamma - 1) * (E / epsilon)**(1 - gamma)

def dFdE(E):
    """Flux on the ground."""
    return C * np.pi * dJdE(E)

def N(E):
    return beta * (E / epsilon)**eta

def rho(r, E, lmbda):
    return N(E) * np.exp(-r / lmbda) / (2 * np.pi * lmbda**2)

def E_min(phid):
    return epsilon * (2 * np.pi * lmbda**2 * phid / beta)**(1 / eta)

def r_D(E, phid):
    """Zero below E_min."""
    return lmbda * eta * np.log(np.maximum(E / E_min(phid), 1.0))

def A_D(E, phid):
    return np.pi * r_D(E, phid)**2

def dRdlnE(E, phid):
    """Rate per unit ln E."""
    return E * dFdE(E) * A_D(E, phid)

def R(phid):
    """The closed form."""
    return 2 * C * np.pi**2 * alpha * (lmbda * eta)**2 * (E_min(phid) / epsilon)**(1 - gamma) / (gamma - 1)**3

In [ ]:
# Spectrum:  the model against the data
E = np.logspace(14, 21, 400)
spec = load("crspectra_spectrum.yaml")
fig, ax = plt.subplots(figsize=(6, 4.5))
for (name, d), mk in zip(spec.items(), "os^vD<>ph"):
    ax.errorbar(d["E"], d["EdJdE"], yerr=[d["lo"], d["hi"]], fmt=mk, markersize=3, linewidth=0.8, label=name)
ax.plot(E, E * dJdE(E), "k-", label="model")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("E (eV)"); ax.set_ylabel("E dJ/dE (1/m$^2$ sr s)")
ax.legend(fontsize=7); plt.tight_layout(); plt.savefig("pdf/analytic_spectrum.pdf"); plt.show()

In [ ]:
# Muon number:  the model against CORSIKA
mu = load("nmuons.yaml")
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.fill_between(mu["E"], mu["p16"], mu["p84"], color="0.85", label="CORSIKA, 16-84%")
ax.plot(mu["E"], mu["median"], ".", color="0.4", markersize=3, label="CORSIKA, median")
ax.plot(E, N(E), label="model")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("E (eV)"); ax.set_ylabel("N (muons)")
ax.legend(); plt.tight_layout(); plt.savefig("pdf/analytic_muon_number.pdf"); plt.show()

In [ ]:
# Lateral profile
r = np.linspace(0, 8 * lmbda, 400)
fig, ax = plt.subplots(figsize=(7, 4.5))
for E in (1e17, 1e18, 1e19, 1e20):
    p = ax.plot(r, rho(r, E, lmbda), label=f"E = {E:.0e} eV")
    ax.plot(r, rho(r, E, 200), "--", color=p[0].get_color())
#for phid in phi_D:
#    ax.axhline(phid, color="0.5", linewidth=0.8)
ax.set_yscale("log"); ax.set_ylim(1, None)
ax.set_xlabel("r (m)"); ax.set_ylabel("muon density (1/m$^2$)")
ax.plot([], [], "k-", label=f"$\\lambda$ = {lmbda:g} m")
ax.plot([], [], "k--", label="$\\lambda$ = 200 m")
ax.set_title("Exponential lateral profile")
ax.legend(); plt.tight_layout(); plt.savefig("pdf/analytic_lateral_profile.pdf"); plt.show()

In [ ]:
# Detectable radius:

E = np.logspace(16, 21, 400)
fig, ax = plt.subplots(figsize=(7, 4.5))
for phid in phi_D:
    above = E > E_min(phid)
    ax.plot(E[above], r_D(E[above], phid), label=f"$\\Phi_D$ = {phid:g}")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("E (eV)"); ax.set_ylabel("$r_D$ (m)")
ax.set_title("Detectable radius")
ax.legend(); plt.tight_layout(); plt.savefig("pdf/analytic_detectable_radius.pdf"); plt.show()

In [ ]:
# Rate: where it comes from; the closed form against the numerical integral
fig, ax = plt.subplots(figsize=(7, 4.5))
for phid in phi_D:
    E = np.logspace(np.log10(E_min(phid)), 21, 400)
    y = dRdlnE(E, phid) * np.log(10) * YEAR
    p = ax.plot(E, y, label=f"$\\Phi_D$ = {phid:g}")
    i = np.argmax(y)
    ax.annotate(f"R = {R(phid) * YEAR:.1f} / yr", xy=(E[i], y[i]), xytext=(60, 25), textcoords="offset points",
                color=p[0].get_color(), arrowprops=dict(arrowstyle="->", color=p[0].get_color()))
ax.set_xscale("log"); ax.set_ylim(0, 1.3 * ax.get_ylim()[1])
ax.set_title("Detectable rate")
ax.set_xlabel("E (eV)"); ax.set_ylabel("dR / dlog$_{10}$E (per year per decade)")
ax.legend(); plt.tight_layout(); plt.savefig("pdf/analytic_rate.pdf"); plt.show()

print(f"{'Phi_D':>6s}  {'E_min (eV)':>10s}  {'rate / yr':>9s}  {'numerical':>9s}")
for phid in phi_D:
    E = np.logspace(np.log10(E_min(phid)), np.log10(E_min(phid)) + 8, 20001)
    num = np.trapezoid(dRdlnE(E, phid), np.log(E))
    print(f"{phid:6g}  {E_min(phid):10.3g}  {R(phid) * YEAR:9.2f}  {num * YEAR:9.2f}")

g = (gamma - 1) / eta
print(f"\nR ~ lambda^{2 - 2 * g:.2f} beta^{g:.2f} Phi_D^{-g:.2f}")